# Grokking-Geometric Kaggle Runner

Use this notebook on Kaggle with **Accelerator = GPU T4 x2** and **Internet = On**.

Recommended flow:

1. Run setup cells.
2. Run the pilot once.
3. Run one full-sweep dataset chunk per Kaggle session.
4. Save `runs_kaggle_full.tar.gz` after each session.
5. In the next session, restore that archive and continue with `--resume`.


## 0. Kaggle Settings

Before running:

- Settings -> Accelerator -> **GPU T4 x2**
- Settings -> Internet -> **On**
- Optional: attach a previous output archive as a Kaggle Dataset if resuming from another session.


In [ ]:
# EDIT THIS FIRST
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
BRANCH = "master"

# Same run root across sessions lets --resume skip finished configs.
RUN_ROOT = "runs_kaggle_full"

# If you attach a previous Kaggle Dataset containing runs_kaggle_full.tar.gz,
# set this to the archive path, for example:
# RESTORE_ARCHIVE = "/kaggle/input/my-grokking-runs/runs_kaggle_full.tar.gz"
RESTORE_ARCHIVE = ""


In [ ]:
!nvidia-smi
!python --version
!df -h /kaggle/working

## 1. Clone Repo

In [ ]:
import os, subprocess, pathlib, shutil

repo_dir = pathlib.Path('/kaggle/working/grokking-geometric')
if not repo_dir.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo_dir)])
else:
    print('Repo already exists:', repo_dir)

os.chdir(repo_dir)
print('cwd =', os.getcwd())
!git log -1 --oneline

## 2. Restore Previous Progress Optional

Use this if you uploaded `runs_kaggle_full.tar.gz` from a previous Kaggle session as an input Dataset.

In [ ]:
import os, subprocess

if RESTORE_ARCHIVE:
    assert os.path.exists(RESTORE_ARCHIVE), RESTORE_ARCHIVE
    print('Restoring:', RESTORE_ARCHIVE)
    subprocess.check_call(['tar', '-xzf', RESTORE_ARCHIVE, '-C', '/kaggle/working/grokking-geometric'])
else:
    print('No restore archive configured.')

## 3. Pilot Run

Run this first. It validates install, datasets, training, analysis, SVCCA, interventions, plotting, and aggregation.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/grokking-geometric
bash run_fresh_linux.sh --mode pilot --run-root runs_kaggle_pilot --system-python --skip-torch-install


## 4. Full Sweep, One Dataset Chunk On T4 x2

Run one dataset per Kaggle session. Change `DATASET` each time:

- `modular_addition`
- `modular_multiplication`
- `symmetric_group`

This cell uses both T4 GPUs by running two independent processes split by seed.

In [ ]:
# EDIT THIS PER SESSION
DATASET = "modular_addition"  # modular_addition, modular_multiplication, symmetric_group
from pathlib import Path
Path('/kaggle/working/grokking-geometric/.kaggle_dataset').write_text(DATASET)
print(DATASET)

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/grokking-geometric
mkdir -p logs

DATASET=$(cat .kaggle_dataset 2>/dev/null || echo modular_addition)
echo "Dataset: ${DATASET}"

CUDA_VISIBLE_DEVICES=0 bash run_fresh_linux.sh \
  --mode full \
  --dataset "${DATASET}" \
  --seeds "0,1,2" \
  --run-root runs_kaggle_full \
  --system-python \
  --skip-torch-install \
  --resume > "logs/gpu0_${DATASET}.log" 2>&1 &

CUDA_VISIBLE_DEVICES=1 bash run_fresh_linux.sh \
  --mode full \
  --dataset "${DATASET}" \
  --seeds "3,4" \
  --run-root runs_kaggle_full \
  --skip-install \
  --resume > "logs/gpu1_${DATASET}.log" 2>&1 &

wait
python summarize_runs.py --runs-dir runs_kaggle_full --output-dir results/runs_kaggle_full


## 5. Progress Check

In [ ]:
import pathlib

run_root = pathlib.Path('/kaggle/working/grokking-geometric') / RUN_ROOT
done = len(list(run_root.glob('*/signals.csv'))) if run_root.exists() else 0
total = 240
print(f'{done}/{total} complete ({100*done/total:.1f}%)')

for p in sorted((pathlib.Path('/kaggle/working/grokking-geometric') / 'logs').glob('*.log')):
    print(p.name, p.stat().st_size, 'bytes')

In [ ]:
!tail -n 80 logs/gpu0_{DATASET}.log || true
!tail -n 80 logs/gpu1_{DATASET}.log || true

## 6. Save Results Archive

Run this after each session. Download the `.tar.gz` from Kaggle output files or save it as a Kaggle Dataset, then restore it in the next session.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/grokking-geometric
python summarize_runs.py --runs-dir runs_kaggle_full --output-dir results/runs_kaggle_full || true
tar -czf /kaggle/working/runs_kaggle_full.tar.gz runs_kaggle_full results/runs_kaggle_full logs
ls -lh /kaggle/working/runs_kaggle_full.tar.gz


## 7. Optional Smaller Width/Seed Chunks

If the dataset chunk above is still too long, run narrower chunks like these instead.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/grokking-geometric
mkdir -p logs

CUDA_VISIBLE_DEVICES=0 bash run_fresh_linux.sh \
  --mode full \
  --dataset modular_addition \
  --seeds "0,1" \
  --widths "64,128" \
  --run-root runs_kaggle_full \
  --system-python \
  --skip-torch-install \
  --resume > logs/gpu0_small_chunk.log 2>&1 &

CUDA_VISIBLE_DEVICES=1 bash run_fresh_linux.sh \
  --mode full \
  --dataset modular_addition \
  --seeds "2,3,4" \
  --widths "64,128" \
  --run-root runs_kaggle_full \
  --skip-install \
  --resume > logs/gpu1_small_chunk.log 2>&1 &

wait


## 8. Optional Tiny Ablation Chunk

Do not run the full 2160-run ablation sweep on free Kaggle. Use tiny chunks only.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/grokking-geometric
bash run_fresh_linux.sh \
  --mode ablation \
  --dataset modular_addition \
  --layers "1" \
  --widths "64,128" \
  --seeds "0" \
  --weight-decays "0.0,1.0" \
  --train-fractions "0.25,1.0" \
  --run-root runs_kaggle_ablation_tiny \
  --system-python \
  --skip-torch-install \
  --resume
